In [1]:
import sys
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import(accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
RANDOM_STATE = 42

In [14]:
def std_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['Gender_Male'] = (df['Gender'] == 'Male').astype(int)
    dept_dummies = pd.get_dummies(df['Department'], prefix='Dept', drop_first=True).astype(int)
    df['High_Stress'] = (df['Stress_Level']>=8).astype(int)
    df=pd.concat([df, dept_dummies], axis=1)
    return df

In [24]:
def main(path=None):
    if path is None:
        if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
            path = sys.argv[1]
        else:
            path = "student_lifestyle_100k.csv"
 
    df = std_data(path)
 

    feature_cols = ["Sleep_Duration", "Study_Hours", "Social_Media_Hours", "Physical_Activity", "Stress_Level", "High_Stress", "CGPA", "Age", "Gender_Male",
                    "Dept_Business", "Dept_Engineering", "Dept_Medical", "Dept_Science",]
    X = df[feature_cols]
    y = df['Depression'].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train_s, y_train)

    y_pred = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:,1]

    metrics = {"accuracy": accuracy_score(y_test, y_pred), "precision": precision_score(y_test, y_pred), "recall": recall_score(y_test, y_pred),
               "f1": f1_score(y_test, y_pred), "roc-auc": roc_auc_score(y_test, y_proba),}
    cm = confusion_matrix(y_test, y_pred)

    print("---- Test set mertics ----")
    for k, v in metrics.items():
        print(f"{k:>10}: {v:.4f}")
    print("\nConfusion matrix [[TN FP]/[FN TP]]:")
    print(cm)

    print("\n----Standardized logistic regression coefficients----")
    for name, coef in zip(feature_cols, model.coef_[0]):
        print(f"{name:>20}: {coef:+.4f}")
    print(f"{'intercept':>20}: {model.intercept_[0]:+.4f}")
 
    print("\n=== Standardization params (mean, scale) per feature ===")
    for name, mean, scale in zip(feature_cols, scaler.mean_, scaler.scale_):
        print(f"{name:>20}: mean={mean:.5f}  scale={scale:.5f}")
 

    export = { "feature_order": feature_cols, "coefficients": model.coef_[0].round(5).tolist(),
                   "intercept": round(float(model.intercept_[0]), 5), "scaler_mean": np.round(scaler.mean_, 5).tolist(),
                   "metrics": {k: round(v, 4) for k, v in metrics.items()}, "confusion_matrix": cm.tolist(), "n_rows": int(len(df)),
                   "n_test": int(len(y_test)), "base_rate": round(float(y.mean()), 4),}
    with open("model_export.json", "w") as f:
        json.dump(export, f, indent=2)
    print("\nWrote model_export.json (paste these exact numbers into index.html)")
         

In [25]:
if __name__ == "__main__":
    main()

---- Test set mertics ----
  accuracy: 0.6371
 precision: 0.1712
    recall: 0.6784
        f1: 0.2734
   roc-auc: 0.6926

Confusion matrix [[TN FP]/[FN TP]]:
[[11378  6610]
 [  647  1365]]

----Standardized logistic regression coefficients----
      Sleep_Duration: -0.0342
         Study_Hours: -0.0071
  Social_Media_Hours: +0.0000
   Physical_Activity: +0.0030
        Stress_Level: +0.0497
         High_Stress: +0.2650
                CGPA: -0.6215
                 Age: -0.0202
         Gender_Male: +0.0093
       Dept_Business: -0.0007
    Dept_Engineering: -0.0092
        Dept_Medical: -0.0209
        Dept_Science: -0.0257
           intercept: -0.1979

=== Standardization params (mean, scale) per feature ===
      Sleep_Duration: mean=6.99785  scale=1.49758
         Study_Hours: mean=4.50996  scale=1.97335
  Social_Media_Hours: mean=3.50010  scale=1.48671
   Physical_Activity: mean=74.29520  scale=43.34253
        Stress_Level: mean=4.12916  scale=1.42361
         High_Stress: mea